# Starter notebook — How to load the data, make a submission, submit it

This notebook is about the **mechanics** of the competition, nothing else:

1. **Find** the data files Kaggle attached to your session, in `/kaggle/input/`.
2. **Load** them and look at a few rows with `df.head(5)`.
3. **Write** a `submission.csv` in the format the competition expects.
4. **Submit** it.

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed.
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python

import os
import numpy as np                 # linear algebra
import pandas as pd                # data processing, CSV / parquet I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory.
# This lists every file the competition attached to your session.
for dirname, _, filenames in os.walk("/kaggle/input"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/), which is preserved as
# output when you create a version using "Save & Run All".


## Load the data

| File | What it is |
|---|---|
| `replication.parquet` | Not graded — a clean panel from the paper's own specification, plus `refcheck.json` as its answer key. Start here. |
| `set_A.parquet`, `set_B.parquet`, `set_C.parquet` | The three graded panels. |
| `refcheck.json` | Reference numbers for `replication.parquet` only. |
| `klw_first_stage.npz` | Innovation pools for forward simulation. |
| `sample_submission.csv` | The 36 required `Id`s. |

One row per surviving **bank-quarter**; `failed = 1` is closure, which is **absorbing** (the
bank leaves the panel afterwards). `estimated_cost` is observed **only for closed banks**.

In [ ]:
# The competition data lives in /kaggle/input/<competition-slug>/. Find it by looking for a
# file we know is there, so you never have to hard-code the slug.
import glob

hits = glob.glob("/kaggle/input/**/set_A.parquet", recursive=True) or glob.glob("**/set_A.parquet", recursive=True)
DATA_DIR = os.path.dirname(hits[0])
OUT_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

print("data  :", DATA_DIR)
print("output:", OUT_DIR)


In [ ]:
# Load the four panels.
panels = {name: pd.read_parquet(os.path.join(DATA_DIR, f)) for name, f in
          [("replication", "replication.parquet"), ("A", "set_A.parquet"),
           ("B", "set_B.parquet"),                 ("C", "set_C.parquet")]}

for name, d in panels.items():
    print(f"{name:12s} {len(d):>7,} rows x {d.shape[1]:2d} cols   "
          f"{d.rssd_id.nunique():,} banks   {int(d.failed.sum()):,} closures")


### A look at the rows

In [ ]:
df = panels["A"]          # switch to "B", "C" or "replication"
df.head(5)


In [ ]:
df.describe().T


## The submission

Two columns, `Id` and `Prediction`, 36 rows. The `Id`s are `{A,B,C}_sigma`,
`{A,B,C}_theta_{term}`, and `{A,B,C}_cf_{no_political,myopic,npl_stress}`.

The easiest valid submission is `sample_submission.csv` itself — all zeros. It scores badly,
but it proves your pipeline end to end. **Do this before you write any model code**, so a
format error costs you nothing instead of a submission.

In [ ]:
sample = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))
print(sample.shape)
sample.head(12)


In [ ]:
# The dummy submission: the 36 required Ids, all predictions zero.
sample.to_csv(os.path.join(OUT_DIR, "submission.csv"), index=False)   # index=False matters

check = pd.read_csv(os.path.join(OUT_DIR, "submission.csv"))
print(f"wrote {OUT_DIR}/submission.csv  —  {len(check)} rows, columns {list(check.columns)}")


## How to submit

**From this notebook:** click **Save Version → Save & Run All (Commit)**. When it finishes,
open the version, go to the **Output** tab, and click **Submit**. Kaggle picks up whatever
`/kaggle/working/submission.csv` contains.

**By hand:** download `submission.csv` from the Output tab and upload it on the competition's
**Submit Predictions** page.

Then replace those zeros: estimate the monetary cost, the state transitions, and the
first-stage CCP; forward-simulate; recover $(\sigma, \theta)$ by GMM; and re-solve the model
under each counterfactual.